In [1]:
import re
from pathlib import Path

import pandas as pd

In [2]:
# version = "eval_4"
# version = "eval_5"
version = "eval_6"
paths = sorted(Path("output/input_space_v2").rglob(f"*/{version}/*/eval_table.csv"))
print(len(paths))

tables = []
for path in paths:
    table = pd.read_csv(path)
    key = path.parts[-4]
    match = re.search(r"(schaefer400|mni_cortex|flat)_lr([-0-9e]+)_([0-9])", key)
    space, lr, trial = match.groups()
    table.insert(0, "trial", int(trial))
    table.insert(0, "base_lr", float(lr))
    table.insert(0, "space", space)
    table.insert(0, "key", key)
    tables.append(table)
table = pd.concat(tables, ignore_index=True)
print(table.shape)
table.head()

48
(168, 19)


,key,space,base_lr,trial,model,repr,clf,dataset,epoch,lr,wd,hparam_id,hparam,split,loss,acc,acc_std,f1,f1_std
0,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,hcpya_task21,19,0.002130,0.05,36,"[7.1, 1.0]",train,0.000063,1.000000,0.000000,1.000000,0.000000
1,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,hcpya_task21,19,0.002130,0.05,36,"[7.1, 1.0]",validation,0.062418,0.988343,0.001733,0.987209,0.002122
2,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,hcpya_task21,19,0.002130,0.05,36,"[7.1, 1.0]",test,0.082430,0.985516,0.001543,0.982407,0.002081
3,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,nsd_cococlip,19,0.000132,0.05,19,"[0.44, 1.0]",train,1.955687,0.414303,0.002398,0.358911,0.002630
4,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,nsd_cococlip,19,0.000132,0.05,19,"[0.44, 1.0]",validation,2.421332,0.280915,0.005559,0.225250,0.005038


In [3]:
summary = table.loc[table["split"] == "test"].pivot_table(
    values=["acc", "acc_std"], index=["space", "base_lr", "trial", "repr", "clf"], columns="dataset"
)
summary = summary.loc[(slice(None), slice(None), slice(None), "patch", "attn")].dropna(
    axis=1, how="all"
)
summary

acc                   acc_std             
dataset                   hcpya_task21 nsd_cococlip hcpya_task21 nsd_cococlip
space       base_lr trial                                                    
flat        0.0010  1         0.985516     0.303711     0.001543     0.005252
                    2         0.985516     0.311132     0.001633     0.005164
                    3         0.988690     0.299814     0.001514     0.005239
                    4         0.986508     0.306679     0.001591     0.005479
                    5         0.986905     0.285343     0.001544     0.005184
                    6         0.986111     0.312987     0.001592     0.005169
                    7         0.985317     0.297774     0.001674     0.005283
                    8         0.984921     0.325417     0.001693     0.005775
mni_cortex  0.0010  1         0.959722     0.268646     0.002750     0.005229
                    2         0.957341     0.257514     0.002849     0.004941
                    3         0.957937     0.264378     0.002700     0.005130
                    4         0.956151     0.259555     0.002930     0.004773
                    5         0.960913     0.269017     0.002846     0.005176
                    6         0.958135     0.256215     0.002698     0.005025
                    7         0.962897     0.255844     0.002656     0.004784
                    8         0.957738     0.263451     0.002745     0.004760
schaefer400 0.0003  1         0.976190     0.275325     0.002198     0.005185
                    2         0.973611     0.260482     0.002184     0.005318
                    3         0.974802     0.267904     0.002171     0.005252
                    4         0.975992     0.271058     0.002198     0.005283
                    5         0.975000     0.276252     0.002155     0.005480
                    6         0.978571     0.275325     0.002022     0.005110
                    7         0.973016     0.266419     0.002250     0.004896
                    8         0.974802     0.270686     0.002296     0.005390

In [4]:
DATASET_NAMES = {
    "abide_dx": "ABIDE Dx",
    "abide_age": "ABIDE Age",
    "abide_sex": "ABIDE Sex",
    "adhd200_dx": "ADHD200 Dx",
    "adhd200_sex": "ADHD200 Sex",
    "adni_ad_vs_cn": "ADNI Dx",
    "adni_sex": "ADNI Sex",
    "ppmi_dx": "PPMI Dx",
    "ppmi_sex": "PPMI Sex",
    "ppmi_age": "PPMI Age",
    "hcpya_rest1lr_gender": "HCP-YA Sex",
    "hcpya_rest1lr_age": "HCP-YA Age",
    "hcpya_rest1lr_neofacn": "HCP-YA NEO-N",
    "aabc_sex": "HCP-A Sex",
    "aabc_age": "HCP-A Age",
    "hcpya_task21": "HCP-YA Task21",
    "nsd_cococlip": "NSD COCO24",
}

SPACE_NAMES = {
    "schaefer400": "parcel",
    "flat": "flat",
    "mni_cortex": "volume",
}


def format_acc_std(acc, std):
    """Format accuracy and std as 'XX.XX ± YY.YY'"""
    if pd.isna(acc) or pd.isna(std):
        return "—"
    return f"{acc * 100:.1f} ± {std * 100:.1f}"
    # return f"{acc * 100:.1f} \mypm{{{std * 100:.1f}}}"

In [5]:
# datasets = ["hcpya_rest1lr_gender", "aabc_sex", "hcpya_task21", "nsd_cococlip"]
# datasets = ["aabc_age", "hcpya_rest1lr_gender", "hcpya_task21", "nsd_cococlip"]
datasets = ["hcpya_task21", "nsd_cococlip"]

space_lrs = {
    "schaefer400": 3e-4,
    "flat": 1e-3,
    "mni_cortex": 1e-3,
}

records = []

for (space, base_lr, trial), row in summary.iterrows():
    if base_lr != space_lrs[space]:
        continue
    record = {"space": space, "lr": base_lr, "trial": trial}
    for ds in datasets:
        acc = row[("acc", ds)]
        std = row[("acc_std", ds)]
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "$\mypm$"))

| space       |     lr |   trial | HCP-YA Task21   | NSD COCO24   |
|:------------|-------:|--------:|:----------------|:-------------|
| flat        | 0.001  |       1 | 98.6 ± 0.2      | 30.4 ± 0.5   |
| flat        | 0.001  |       2 | 98.6 ± 0.2      | 31.1 ± 0.5   |
| flat        | 0.001  |       3 | 98.9 ± 0.2      | 30.0 ± 0.5   |
| flat        | 0.001  |       4 | 98.7 ± 0.2      | 30.7 ± 0.5   |
| flat        | 0.001  |       5 | 98.7 ± 0.2      | 28.5 ± 0.5   |
| flat        | 0.001  |       6 | 98.6 ± 0.2      | 31.3 ± 0.5   |
| flat        | 0.001  |       7 | 98.5 ± 0.2      | 29.8 ± 0.5   |
| flat        | 0.001  |       8 | 98.5 ± 0.2      | 32.5 ± 0.6   |
| mni_cortex  | 0.001  |       1 | 96.0 ± 0.3      | 26.9 ± 0.5   |
| mni_cortex  | 0.001  |       2 | 95.7 ± 0.3      | 25.8 ± 0.5   |
| mni_cortex  | 0.001  |       3 | 95.8 ± 0.3      | 26.4 ± 0.5   |
| mni_cortex  | 0.001  |       4 | 95.6 ± 0.3      | 26.0 ± 0.5   |
| mni_cortex  | 0.001  |       5 | 96.1 ± 0.3   

In [6]:
def sem(x):
    return x.std() / (len(x) ** 0.5)


logistic_summary = table.loc[(table["split"] == "test") & (table["clf"] == "logistic")].pivot_table(
    values=["acc"],
    index=["space", "base_lr", "trial", "repr", "clf"],
    columns="dataset",
    aggfunc=["mean", sem],
)
logistic_summary = logistic_summary.loc[
    (slice(None), slice(None), slice(None), "cls", "logistic")
].dropna(axis=1, how="all")
logistic_summary

KeyError: 'cls'

In [ ]:
# datasets = ["abide_dx", "adhd200_dx", "adni_ad_vs_cn", "ppmi_dx", "aabc_age", "aabc_sex", "hcpya_rest1lr_gender"]
datasets = ["adni_ad_vs_cn", "ppmi_dx", "aabc_age", "aabc_sex", "hcpya_rest1lr_gender"]

space_lrs = {
    "schaefer400": 3e-4,
    "flat": 1e-3,
    "mni_cortex": 1e-3,
}

records = []

for (space, base_lr, trial), row in logistic_summary.iterrows():
    if base_lr != space_lrs[space]:
        continue
    record = {"space": space, "lr": base_lr, "trial": trial}
    for ds in datasets:
        acc = row[("mean", "acc", ds)]
        std = row[("sem", "acc", ds)]
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "$\mypm$"))

| space       |     lr |   trial | ADNI Dx    | PPMI Dx    | HCP-A Age   | HCP-A Sex   | HCP-YA Sex   |
|:------------|-------:|--------:|:-----------|:-----------|:------------|:------------|:-------------|
| flat        | 0.001  |       1 | 76.8 ± 0.3 | 63.4 ± 0.3 | 40.8 ± 0.3  | 87.7 ± 0.2  | 86.9 ± 0.2   |
| flat        | 0.001  |       2 | 75.8 ± 0.2 | 62.2 ± 0.2 | 44.7 ± 0.3  | 87.0 ± 0.3  | 87.7 ± 0.2   |
| flat        | 0.001  |       3 | 76.0 ± 0.2 | 64.0 ± 0.2 | 42.1 ± 0.3  | 86.9 ± 0.2  | 84.7 ± 0.3   |
| flat        | 0.001  |       4 | 76.7 ± 0.2 | 62.7 ± 0.3 | 43.6 ± 0.3  | 87.5 ± 0.2  | 87.9 ± 0.2   |
| flat        | 0.001  |       5 | 76.0 ± 0.2 | 63.0 ± 0.2 | 43.9 ± 0.3  | 87.3 ± 0.2  | 86.8 ± 0.2   |
| mni_cortex  | 0.001  |       1 | 77.2 ± 0.2 | 63.0 ± 0.2 | 52.2 ± 0.3  | 85.5 ± 0.2  | 90.9 ± 0.2   |
| mni_cortex  | 0.001  |       2 | 77.0 ± 0.2 | 63.0 ± 0.2 | 53.5 ± 0.3  | 85.2 ± 0.2  | 90.1 ± 0.2   |
| mni_cortex  | 0.001  |       3 | 75.6 ± 0.2 | 63.9 ± 0.2 | 52.